<!--
File: technical_memory.ipynb
Description: Running log of technical decisions, tradeoffs, and gotchas for the josejorge/josejorge GitHub profile README repo.
Author: Jose-Jorge HERNANDEZ
Company: Parlee Conseiller, Inc.
Date: 2026-09-14
Last edit date: 2026-09-14
Version: 3.0.0
-->

# Technical Memory — josejorge/josejorge

This notebook is the *why* behind technical decisions in this repo — not a changelog (that's `git log`) and not a bug log. Each entry below is dated and explains reasoning, tradeoffs weighed, and gotchas hit.

## 2026-09-14 — Activity Graph section was pointing at a dead service

**Symptom:** the README's "📈 Activity Graph" section never rendered.

**Root cause:** it embedded a live `<img>` from `https://github-readme-activity-graph.vercel.app/graph?...`. That public Vercel deployment has been shut down entirely — it returns HTTP 402 with body `DEPLOYMENT_DISABLED`, not a transient outage. Confirmed by direct `curl` below.

**Decision:** rather than embedding *any* live third-party URL (the same class of risk that caused this), follow the repo's own established pattern — already used for `assets/dashboard.png` (Playwright screenshot of the Grafana dashboard, committed every 6h) and `assets/top-langs.svg` (fetched via a GitHub Action, committed) — of generating the asset server-side in CI and committing a static file, so README always renders something even if the generating service is slow or briefly down.

**Source swap:** `ghchart.rshah.org` was chosen as the new source (a long-stable, actively-maintained contribution-heatmap SVG generator) instead of trying to resurrect or self-host `github-readme-activity-graph`. `.github/workflows/stats.yml` now has a `curl --fail` step that fetches `assets/activity-graph.svg` alongside the existing top-langs step; `--fail` makes the step error loudly (rather than silently committing an error page) if this source ever goes down too.

In [ ]:
%%bash
# Evidence: the old activity-graph source is dead, not flaky.
curl -sL --max-time 15 -o /dev/null -w "old source (github-readme-activity-graph.vercel.app): HTTP %{http_code}\n" \
  "https://github-readme-activity-graph.vercel.app/graph?username=josejorge&theme=tokyo-night&hide_border=true"

# Evidence: the replacement source is alive and returns a real SVG.
curl -sL --max-time 15 -o /dev/null -w "new source (ghchart.rshah.org): HTTP %{http_code}\n" \
  "https://ghchart.rshah.org/0e75b6/josejorge"

## 2026-09-14 — Merging 3 blogs into "Latest Blog Posts" required treating all 3 feeds as equally Cloudflare-protected

**Context:** `scripts/update-blog-readme.js` already fetched `blog.kythex.com/feed` through a real headless browser (Playwright) because Cloudflare's Bot Fight Mode issues a silent JS challenge to plain HTTP clients from datacenter ASNs — which is exactly what GitHub-hosted Actions runners are. The task was to add `thealzdiary.com/feed` and `tierradeoz.com/feed`.

**Gotcha avoided:** a plain `curl` from a residential/local IP against both new feeds returned `200` with clean RSS XML directly — no visible Cloudflare challenge. It would have been tempting to conclude these two feeds don't need the browser workaround and fetch them with a lighter-weight plain HTTP request in CI.

**Why that would have been wrong:** Cloudflare Bot Fight Mode's challenge targets requests from datacenter/hosting-provider ASNs specifically — a local residential-IP test passing proves nothing about how the *same* request will behave from a GitHub Actions runner's IP range. Since the failure mode (silent challenge page mistaken for the real feed) is hard to detect after the fact, all three feeds are now fetched through the same shared Playwright browser/context in `fetchAllBlogs()`, uniformly, rather than special-casing two of them as "safe" based on a non-representative local test.

**Output shape decision:** the user asked for a 3-column Markdown table (one column per blog) instead of a single interleaved-by-date list, so each blog gets its own column with up to 5 of its most recent posts; a blog with fewer recent posts than another just leaves blank cells in its column rather than another blog's post bleeding into that row.

## 2026-09-14 — "Featured Work" section added, Jireh case study kept anonymized

**Context:** the README led entirely with a KytheX/hacker-terminal/gaming persona before any real engineering proof, which is weak for recruiters skimming the profile. A "Featured Work" section was added near the top, right after the About Me block.

**Decision on scope:** two projects were included — `calendar-appointments` (public repo under the `josejorge` GitHub account, linked directly) and a payroll + notification platform built for a client. The client project's actual repos (`apharenxis/jirehhomecleaning*`) are private and under a different account — not the user's to link publicly. Per explicit instruction, that case study is described generically ("a home-services company", no client name, no repo link) rather than naming the client, even though the client's own public-facing site exists — confidentiality/scope was the deciding factor, not technical constraint.

## 2026-09-14 (2) — Centering a `<table>` needs `align` on the table itself, not a wrapping `<div>`

**Symptom:** the user asked to center the "sections" of the README — specifically clarified as the divs/tables/boxes themselves, not the text inside them. The 3 tables (Featured Work, Current Projects, Latest Blog Posts) sat flush left.

**Why wrapping in `<p align="center">` or `<div align="center">` wouldn't have worked:** that pattern already centers every image in this README, but only because an `<img>` is an *inline* element — `text-align: center` on its block-level parent centers inline content within the parent's full-width box. A `<table>` is block-level with shrink-to-fit width; `text-align` on an ancestor has no effect on where a block-level child's box sits. The only reliable fix is the legacy `align` attribute (or `margin: auto`) on the `<table>` element itself — this is the same trick widely used across other GitHub profile READMEs, and GitHub's HTML sanitizer allows it.

**Consequence:** the Current Projects list and the blog-posts table had to be converted from Markdown pipe-table syntax to raw HTML `<table>` markup, because Markdown's table syntax gives the renderer no way to attach an `align` attribute to the `<table>` it generates. `scripts/update-blog-readme.js` now emits HTML directly (`buildBlogTable()`) instead of Markdown table rows, for the same reason — every future run needs to keep producing a centered table, not just this one edit.

**Left alone:** the `<details>` Tech Stack cards and the fenced ` ```yaml `/` ```diff ` blocks were not touched — both are block-level elements that already stretch to 100% of the container width by default (no intrinsic shrink-to-fit sizing), so there is no "off-center" position for them to be in; centering them would be a no-op.

## 2026-09-14 (3) — Featured Work enrichment: excluded two local repos because they're forks, not original work

**Context:** asked to enrich "Featured Work" with "more complex examples" from `H:\DEV`. Surveyed ~90 local project directories, checked each candidate's `git remote` to find the real GitHub owner, and used the GitHub API (`GET /repos/{owner}/{repo}` → 200 vs 404) to confirm which ones are actually public before linking them — several looked promising locally (`parlee-nexus-notifycore`, `parlee-sso`, `zhumobile-campaign-system`, `ParleePay`, several GoDGuilD repos) but turned out to be private, so they were left out rather than linked to a 404.

**Gotcha caught before it became a misrepresentation:** two of the strongest-*looking* candidates by README polish — `H:\DEV\waveterm` and `H:\DEV\tabularis` — are local forks of large, well-known open-source projects (Wave Terminal by wavetermdev, and Tabularis by TabularisDB), not original work. Their `git log` history is entirely upstream commits (dependency bumps, upstream security patches) with nothing authored by the user. Featuring either as "his work" on a recruiter-facing profile would overstate authorship of someone else's project. Both were excluded.

**What was included instead:** `browser-picker-pro` (ParleeConseiller org) and `tabularis_cloudflare_d1_plugin` (josejorge) were verified as genuinely original — the former's README says "Main developer: Jose-Jorge HERNANDEZ" outright, and the latter is a real plugin *for* the Tabularis fork above (Rust, talks to Cloudflare D1's REST API through Tabularis' plugin architecture) — a legitimate, honestly-scoped claim ("I built a plugin for this open-source app") rather than a dishonest one ("I built this app").

## 2026-09-14 (4) — Discovered Parlee Conseiller is a multi-brand holding company, but kept anonymizing anyway

**Context:** asked to move "Featured Work" near the end of the README and include many private repos as anonymized case studies, "like Jireh was", using collapsible lists to avoid bloat.

**Finding that could have changed the approach:** reading the private repos' own READMEs revealed that `apharenxis` (the account hosting `cyperzax`, `tenain_platform`, etc.) and the various `parlee-*` repos are not third-party client work at all — they're business units of the user's own multi-brand company, Parlee Conseiller (confirmed via `parlee-contact-hub`'s README, which lists Apharenxis, CyperZaX, EnclavE, GoDGuilD, MeXpresate, TIRNANOK, Ventura Ligia, ZhuMobile, and Ancelot as sibling brands under one Parlee-run contact form). Several of these brands have their own public-facing marketing sites (tirnanok.com, etc.), so naming them wouldn't have been a confidentiality problem the way an actual third-party client's business would be.

**Decision:** anonymized them anyway, per explicit instruction ("obviously anonymous just as Jireh was"). The one genuine third-party client in the set — Jireh Home Cleaning — was already anonymized in the prior pass and needed no re-work; it's grouped under "Client Systems" alongside the other anonymized entries so the section reads uniformly rather than mixing named-brand cards with anonymous ones.

**Rate-limit note:** the unauthenticated GitHub API sweep (`GET /repos/{owner}/{repo}` across ~80 local repos to sort public vs. private) hit the 60-req/hour unauthenticated cap partway through — the last ~15 repos checked all came back `403`, which is a rate-limit response, not evidence of anything about those repos. They weren't used as a basis for any decision in this pass.

**Structure decision:** rather than one long collapsible list, case studies were split into 3 themed `<details>` groups (Client Systems / Internal Platforms / Identity & Security), each holding a compact 3-column table (case study · what it does · stack) with one-line descriptions — mirrors the existing Tech Stack section's per-category collapsible pattern, and keeps each entry skimmable per the user's separate ask to keep case-study text concise.

## 2026-09-21 - Substack added as a 4th blog column

**Decision:** reuse the existing Playwright-based fetch path for Substack instead of a plain `fetch`/`curl` step.

**Why:** a plain request to `https://josejorgehz.substack.com/feed` returned HTTP 200 with valid RSS from a residential IP, but the other three feeds also do, and they get JS-challenged from GitHub runners (see the Cloudflare note in `blog-posts.yml`). Substack sits behind a CDN too, so keeping one uniform browser path avoids a second code path and a possible datacenter-IP surprise.

**Gotcha:** Substack wraps titles in CDATA and its `<item>` shape matches WordPress's (`<title>`, `<link>`), so `parsePosts` works unchanged. The publication currently has one post, so the column is mostly blank; `buildBlogTable` already tolerates short columns.
